In [3]:
import xarray as xr
import numpy as np
import pandas as pd
from pathlib import Path
import os
import subprocess

# Common mount points at GIUB
possible_paths = [
    "/mnt/hydroshare/data/Meteorology/HMA/HARv2_d10km_h_2d_t2_1984.nc",
    "/media/hydroshare/data/Meteorology/HMA/HARv2_d10km_h_2d_t2_1984.nc",
    "/run/user/1000/gvfs/smb-share:server=hydroshare.giub.unibe.ch,share=data/Meteorology/HMA/HARv2_d10km_h_2d_t2_1984.nc",
    "~/hydroshare/data/Meteorology/HMA/HARv2_d10km_h_2d_t2_1984.nc",
    # Additional GVFS patterns (auto-mounted network drives)
    f"/run/user/{os.getuid()}/gvfs/smb-share:server=hydroshare.giub.unibe.ch,share=data/Meteorology/HMA/HARv2_d10km_h_2d_t2_1984.nc",
]

# Also check for any GVFS mount dynamically
try:
    gvfs_path = Path(f"/run/user/{os.getuid()}/gvfs")
    if gvfs_path.exists():
        for mount in gvfs_path.iterdir():
            if 'hydroshare' in mount.name.lower():
                possible_file = mount / "Meteorology/HMA/HARv2_d10km_h_2d_t2_1984.nc"
                if possible_file.exists():
                    possible_paths.insert(0, str(possible_file))
except Exception:
    pass

print("🔍 Searching for HAR file in common locations...")

# Check which path exists
har_file = None
for path in possible_paths:
    expanded_path = os.path.expanduser(path)
    if os.path.exists(expanded_path):
        har_file = expanded_path
        print(f"✅ Found file at: {har_file}")
        break

if har_file is None:
    print("❌ File not found at any common mount point!")
    print("\n🔧 Debugging information:")
    print("\nChecking mounted filesystems...")
    result = subprocess.run(['mount'], capture_output=True, text=True)
    smb_mounts = [line for line in result.stdout.split('\n') if 'hydroshare' in line.lower() or 'gvfs' in line.lower()]
    
    if smb_mounts:
        print("\n🔍 Found potential mounts:")
        for mount in smb_mounts:
            print(f"   {mount}")
    
    # Check GVFS mounts
    gvfs_base = Path(f"/run/user/{os.getuid()}/gvfs")
    if gvfs_base.exists():
        print(f"\n📁 GVFS mounts in {gvfs_base}:")
        for item in gvfs_base.iterdir():
            print(f"   {item.name}")
    
    print("\n💡 To access the file:")
    print("   1. Open your file manager (Files/Nautilus)")
    print("   2. Click on 'Other Locations' in sidebar")
    print("   3. Connect to: smb://hydroshare.giub.unibe.ch/data")
    print("   4. Browse to: Meteorology/HMA/")
    print("   5. Re-run this script - it should auto-detect the mount")
    
else:
    # File found - proceed with analysis
    print(f"\n📁 Opening HAR temperature file...")
    print(f"   File: {har_file}")

    # Open the NetCDF file
    ds = xr.open_dataset(har_file)

    print("\n" + "="*80)
    print("📊 DATASET STRUCTURE")
    print("="*80)

    # Display basic info
    print(f"\nDataset dimensions:")
    for dim, size in ds.dims.items():
        print(f"   {dim}: {size}")

    print(f"\nDataset coordinates:")
    for coord in ds.coords:
        print(f"   {coord}: {ds.coords[coord].shape} - {ds.coords[coord].dtype}")
        if hasattr(ds.coords[coord], 'long_name'):
            print(f"      Long name: {ds.coords[coord].long_name}")
        if hasattr(ds.coords[coord], 'units'):
            print(f"      Units: {ds.coords[coord].units}")

    print(f"\nData variables:")
    for var in ds.data_vars:
        print(f"   {var}: {ds[var].shape} - {ds[var].dtype}")
        if hasattr(ds[var], 'long_name'):
            print(f"      Long name: {ds[var].long_name}")
        if hasattr(ds[var], 'units'):
            print(f"      Units: {ds[var].units}")
        if hasattr(ds[var], 'description'):
            print(f"      Description: {ds[var].description}")

    print(f"\nGlobal attributes:")
    for attr in ds.attrs:
        print(f"   {attr}: {ds.attrs[attr]}")

    print("\n" + "="*80)
    print("📊 DATA SAMPLE")
    print("="*80)

    # Show time range
    if 'time' in ds.coords:
        print(f"\nTime range:")
        print(f"   Start: {pd.Timestamp(ds.time.values[0])}")
        print(f"   End: {pd.Timestamp(ds.time.values[-1])}")
        print(f"   Total timesteps: {len(ds.time)}")
        print(f"   Temporal resolution: {ds.time.values[1] - ds.time.values[0]}")

    # Show spatial extent
    if 'lat' in ds.coords and 'lon' in ds.coords:
        print(f"\nSpatial extent:")
        print(f"   Latitude: {float(ds.lat.min()):.3f}° to {float(ds.lat.max()):.3f}°")
        print(f"   Longitude: {float(ds.lon.min()):.3f}° to {float(ds.lon.max()):.3f}°")
        print(f"   Grid cells: {len(ds.lat)} x {len(ds.lon)}")

    # Show sample temperature values (if t2 variable exists)
    if 't2' in ds.data_vars:
        print(f"\nTemperature statistics (first timestep):")
        t2_sample = ds['t2'].isel(time=0)
        print(f"   Min: {float(t2_sample.min()):.2f}")
        print(f"   Max: {float(t2_sample.max()):.2f}")
        print(f"   Mean: {float(t2_sample.mean()):.2f}")
        print(f"   Median: {float(t2_sample.median()):.2f}")

    print("\n" + "="*80)
    print("✅ Dataset structure check complete!")
    print("="*80)

    # Close the dataset
    ds.close()

🔍 Searching for HAR file in common locations...
✅ Found file at: /run/user/1001/gvfs/smb-share:server=hydroshare.giub.unibe.ch,share=data/Meteorology/HMA/HARv2_d10km_h_2d_t2_1984.nc

📁 Opening HAR temperature file...
   File: /run/user/1001/gvfs/smb-share:server=hydroshare.giub.unibe.ch,share=data/Meteorology/HMA/HARv2_d10km_h_2d_t2_1984.nc

📊 DATASET STRUCTURE

Dataset dimensions:
   time: 8784
   south_north: 252
   west_east: 381

Dataset coordinates:
   time: (8784,) - datetime64[ns]
      Long name: Time
   west_east: (381,) - float32
      Long name: x-coordinate in Cartesian system
      Units: m
   south_north: (252,) - float32
      Long name: y-coordinate in Cartesian system
      Units: m
   lon: (252, 381) - float32
      Long name: Longitude
      Units: degrees_east
   lat: (252, 381) - float32
      Long name: Latitude
      Units: degrees_north

Data variables:
   t2: (8784, 252, 381) - float32
      Long name: temp at 2 m
      Units: k

Global attributes:
   TITLE: HA

/tmp/ipykernel_26357/1820008716.py:81: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  for dim, size in ds.dims.items():
<frozen _collections_abc>:899: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.


   Min: 237.71
   Max: 295.19
   Mean: 263.98
   Median: 262.46

✅ Dataset structure check complete!


In [6]:
import xarray as xr
import numpy as np
import pandas as pd
from pathlib import Path

# Define the directory and files
har_dir = Path("/home/jberg/OneDrive/Raven_worldwide/01_data/meteo/HAR")
files = [
    "HARv2_d10km_d_2d_potevap_1980.nc",
    "HARv2_d10km_d_2d_prcp_1980.nc",
    "HARv2_d10km_d_2d_t2_max_1980.nc",
    "HARv2_d10km_d_2d_t2_mean_1980.nc",
    "HARv2_d10km_d_2d_t2_min_1980.nc"
]

print("🔍 Checking HAR NetCDF file structures...")
print("="*100)

for filename in files:
    filepath = har_dir / filename
    
    if not filepath.exists():
        print(f"\n❌ File not found: {filename}")
        continue
    
    print(f"\n{'='*100}")
    print(f"📁 FILE: {filename}")
    print(f"{'='*100}")
    
    try:
        # Open the dataset
        ds = xr.open_dataset(filepath)
        
        # File size
        file_size_mb = filepath.stat().st_size / (1024 * 1024)
        print(f"\n💾 File size: {file_size_mb:.1f} MB")
        
        # Dimensions (use .sizes instead of .dims to avoid warning)
        print(f"\n📏 DIMENSIONS:")
        for dim, size in ds.sizes.items():
            print(f"   {dim:15s}: {size:6d}")
        
        # Coordinates
        print(f"\n🗺️  COORDINATES:")
        for coord in ds.coords:
            coord_data = ds.coords[coord]
            print(f"   {coord:15s}: shape={str(coord_data.shape):15s} dtype={coord_data.dtype}")
            
            # Show range for numeric coordinates
            if coord_data.dtype in [np.float32, np.float64, np.int32, np.int64]:
                if len(coord_data) > 0:
                    print(f"      → Range: {float(coord_data.min()):.4f} to {float(coord_data.max()):.4f}")
            
            # Show attributes
            if hasattr(coord_data, 'long_name'):
                print(f"      → Long name: {coord_data.long_name}")
            if hasattr(coord_data, 'units'):
                print(f"      → Units: {coord_data.units}")
        
        # Data Variables
        print(f"\n📊 DATA VARIABLES:")
        for var in ds.data_vars:
            var_data = ds[var]
            print(f"   {var:15s}: shape={str(var_data.shape):15s} dtype={var_data.dtype}")
            
            # Show attributes
            if hasattr(var_data, 'long_name'):
                print(f"      → Long name: {var_data.long_name}")
            if hasattr(var_data, 'units'):
                print(f"      → Units: {var_data.units}")
            if hasattr(var_data, 'description'):
                print(f"      → Description: {var_data.description}")
            
            # Show sample statistics
            try:
                sample = var_data.isel(time=0) if 'time' in var_data.dims else var_data
                valid_data = sample.values[~np.isnan(sample.values)]
                if len(valid_data) > 0:
                    print(f"      → Sample stats (first timestep):")
                    print(f"         Min: {valid_data.min():.4f}, Max: {valid_data.max():.4f}, Mean: {valid_data.mean():.4f}")
            except Exception as e:
                print(f"      → Could not compute stats: {e}")
        
        # Global Attributes
        print(f"\n🌍 GLOBAL ATTRIBUTES:")
        for attr in ds.attrs:
            attr_value = ds.attrs[attr]
            # Truncate long attributes
            if isinstance(attr_value, str) and len(attr_value) > 100:
                attr_value = attr_value[:100] + "..."
            print(f"   {attr:20s}: {attr_value}")
        
        # Time information
        if 'time' in ds.coords:
            print(f"\n⏰ TIME INFORMATION:")
            time_vals = ds.time.values
            print(f"   First timestep: {pd.Timestamp(time_vals[0])}")
            print(f"   Last timestep:  {pd.Timestamp(time_vals[-1])}")
            print(f"   Total steps:    {len(time_vals)}")
            if len(time_vals) > 1:
                dt = pd.Timestamp(time_vals[1]) - pd.Timestamp(time_vals[0])
                print(f"   Time step:      {dt}")
        
        # Spatial information
        if 'lat' in ds.coords and 'lon' in ds.coords:
            print(f"\n🗺️  SPATIAL INFORMATION:")
            print(f"   Latitude:  {float(ds.lat.min()):7.3f}° to {float(ds.lat.max()):7.3f}° ({len(ds.lat)} cells)")
            print(f"   Longitude: {float(ds.lon.min()):7.3f}° to {float(ds.lon.max()):7.3f}° ({len(ds.lon)} cells)")
            
            # Estimate resolution - FIX: properly extract scalar values
            if len(ds.lat) > 1 and len(ds.lon) > 1:
                lat_vals = ds.lat.values
                lon_vals = ds.lon.values
                lat_res = abs(float(lat_vals[1] - lat_vals[0]))
                lon_res = abs(float(lon_vals[1] - lon_vals[0]))
                print(f"   Resolution: ~{lat_res:.4f}° x {lon_res:.4f}° (~{lat_res*111:.1f} km x {lon_res*111:.1f} km)")
        
        # Close dataset
        ds.close()
        
    except Exception as e:
        print(f"\n❌ Error reading file: {e}")
        import traceback
        traceback.print_exc()

print("\n" + "="*100)
print("✅ All files checked!")
print("="*100)

🔍 Checking HAR NetCDF file structures...

📁 FILE: HARv2_d10km_d_2d_potevap_1980.nc

💾 File size: 106.4 MB

📏 DIMENSIONS:
   time           :    366
   south_north    :    252
   west_east      :    381

🗺️  COORDINATES:
   time           : shape=(366,)          dtype=datetime64[ns]
      → Long name: Time
   west_east      : shape=(381,)          dtype=float32
      → Range: -1675001.0000 to 2124999.0000
      → Long name: x-coordinate in Cartesian system
      → Units: m
   south_north    : shape=(252,)          dtype=float32
      → Range: -744999.0000 to 1765001.0000
      → Long name: y-coordinate in Cartesian system
      → Units: m
   lon            : shape=(252, 381)      dtype=float32
      → Range: 61.4748 to 110.0567
      → Long name: Longitude
      → Units: degrees_east
   lat            : shape=(252, 381)      dtype=float32
      → Range: 23.3861 to 47.7842
      → Long name: Latitude
      → Units: degrees_north

📊 DATA VARIABLES:
   potevap        : shape=(366, 252, 381

Traceback (most recent call last):
  File "/tmp/ipykernel_26357/2781401276.py", line 114, in <module>
    lat_res = abs(float(lat_vals[1] - lat_vals[0]))
                  ~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: only length-1 arrays can be converted to Python scalars
Traceback (most recent call last):
  File "/tmp/ipykernel_26357/2781401276.py", line 114, in <module>
    lat_res = abs(float(lat_vals[1] - lat_vals[0]))
                  ~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: only length-1 arrays can be converted to Python scalars
Traceback (most recent call last):
  File "/tmp/ipykernel_26357/2781401276.py", line 114, in <module>
    lat_res = abs(float(lat_vals[1] - lat_vals[0]))
                  ~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: only length-1 arrays can be converted to Python scalars
Traceback (most recent call last):
  File "/tmp/ipykernel_26357/2781401276.py", line 114, in <module>
    lat_res = abs(float(lat_vals[1] - lat_vals[0]))
                  ~~~~~^^^^^^^